# 01 Data Extraction And Audit

This notebook opens the thesis package the way a reviewer should: first by understanding where the dataset came from, what was packaged locally, and how the final modeling subsets were defined.

**Questions answered here**
- What data are inside the thesis package?
- How were hybrid, paired-labeled, and paired-unlabeled subsets defined?
- Why is `recording_key` the leakage boundary?
- What did the extraction and audit phase actually establish before modeling began?


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (
            (candidate / "src" / "qc_thesis" / "__init__.py").exists()
            and (candidate / "README.md").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()


def show_saved_figure(path, caption=None):
    figure_path = Path(path)
    if not figure_path.is_absolute():
        figure_path = (ROOT / figure_path).resolve()
    if not figure_path.exists():
        display(Markdown(f"_Missing figure: `{figure_path}`_"))
        return
    if caption:
        display(Markdown(caption))
    display(Image(filename=str(figure_path)))

fig_dir, table_dir = notebook_output_dirs("01_data_extraction_and_audit")
dataset_summary = load_dataset_summary()
computed_summary = compute_dataset_summary()
partition_table = build_dataset_partition_table()
target_missingness = build_target_missingness_table()
family_coverage = build_family_coverage_table()
feature_inventory = build_feature_inventory_table()
recipe_registry = build_recipe_inventory()
manifest = json.loads((ROOT / "data/frozen_inputs/manifest.json").read_text())


## 1. Packaged assets

The thesis package contains both the exploratory raw parquet and the generated benchmark assets. The point is not only to show the final tables, but to show the entire route from extraction to evaluation.


In [ ]:
display(pd.DataFrame(dataset_summary.items(), columns=["packaged_item", "value"]))
display(pd.DataFrame(computed_summary.items(), columns=["recomputed_item", "value"]))
display(recipe_registry.loc[:, ["recipe_id", "target", "runner_key", "main_text"]])
display(pd.DataFrame(manifest.items(), columns=["artifact_group", "path"]))
save_table(pd.DataFrame(dataset_summary.items(), columns=["item", "value"]), table_dir, "packaged_dataset_summary")
save_table(pd.DataFrame(computed_summary.items(), columns=["item", "value"]), table_dir, "recomputed_dataset_summary")
save_table(recipe_registry, table_dir, "recipe_registry")


## 2. Subset definitions

The thesis uses three operational subsets:

- **hybrid labeled**: large source domain with labels
- **paired labeled**: small matched target-domain supervision
- **paired unlabeled**: real target-domain support used only structurally

This table is the foundation for every later benchmark and ablation.


In [ ]:
display(partition_table)
display(target_missingness)
save_table(partition_table, table_dir, "dataset_partition_table")
save_table(target_missingness, table_dir, "target_missingness")


## 3. Family and feature coverage

Before modeling, the important question was whether the paired domain behaved like one domain or a mixture of regimes. Family coverage and feature inventory already hint that the answer is “mixture”.


In [ ]:
display(family_coverage)
display(feature_inventory)
save_table(family_coverage, table_dir, "paired_family_coverage")
save_table(feature_inventory, table_dir, "feature_inventory")


## 4. Why the split protocol is strict

The benchmark is optimized on **recording-disjoint** transfer. A second stress protocol leaves an entire paired family unseen. This notebook makes that separation explicit before any performance claims are shown.


In [ ]:
bundle = build_dataset_bundle(with_context=False)
split = recording_disjoint_split(bundle.paired_labeled_df, test_rows_target=100, random_state=42)
lofo_rows = []
for family_name in sorted(bundle.paired_labeled_df["study_set"].dropna().unique()):
    family_split = leave_one_family_out(bundle.paired_labeled_df, family=family_name)
    lofo_rows.append({
        "held_out_family": family_name,
        "train_rows": len(family_split.train_df),
        "test_rows": len(family_split.test_df),
        "train_recordings": family_split.train_df["recording_key"].nunique(),
        "test_recordings": family_split.test_df["recording_key"].nunique(),
    })
lofo_table = pd.DataFrame(lofo_rows).sort_values("held_out_family").reset_index(drop=True)
protocol_summary = pd.DataFrame([
    {
        "protocol": "recording_disjoint_main",
        "train_rows": len(split.train_df),
        "test_rows": len(split.test_df),
        "train_recordings": split.train_df["recording_key"].nunique(),
        "test_recordings": split.test_df["recording_key"].nunique(),
    }
])
display(protocol_summary)
display(lofo_table)
save_table(protocol_summary, table_dir, "recording_disjoint_protocol_summary")
save_table(lofo_table, table_dir, "family_held_out_protocol_summary")


In [ ]:
display(Markdown(
    f"""
## Key takeaways

- The thesis package contains **{partition_table.loc[partition_table['subset'] == 'full', 'rows'].iloc[0]:,} rows** in total.
- Only **{partition_table.loc[partition_table['subset'] == 'paired_labeled', 'rows'].iloc[0]} paired labeled rows** are available for direct supervised adaptation.
- The benchmark therefore depends on transfer and conservative use of **{partition_table.loc[partition_table['subset'] == 'paired_unlabeled', 'rows'].iloc[0]:,} paired unlabeled rows**.
- `fmiss` has the strongest missing-label burden, which already suggests that `fmiss` will be scientifically harder than `fpos`.
"""
))
